In [ ]:
# models/compare_models.py

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("../data/youtube_india_clean.csv")

# =========================
# 2. DEFINE FEATURES
# =========================
feature_cols = [
    "video_category_id",
    "video_definition",
    "channel_video_count",
    "channel_have_hidden_subscribers",
    "video_duration_seconds"
]

# Classification target
target_class = "high_performance"

# Regression target
target_reg = "video_view_count"

X = df[feature_cols]
y_class = df[target_class]
y_reg = df[target_reg]

# Separate column types
categorical_cols = [
    "video_category_id",
    "video_definition",
    "channel_have_hidden_subscribers"
]

numeric_cols = [
    "channel_video_count",
    "video_duration_seconds"
]

# =========================
# 3. PREPROCESSING
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# =========================
# 4. CLASSIFICATION MODELS
# =========================
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier()
}

# =========================
# 5. REGRESSION MODELS
# =========================
regression_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42),
    "KNN Regressor": KNeighborsRegressor()
}

# =========================
# 6. TRAIN / TEST SPLIT
# =========================
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# =========================
# 7. RUN CLASSIFICATION
# =========================
classification_results = []

for name, model in classification_models.items():
    clf_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf_pipeline.fit(X_train_c, y_train_c)
    y_pred = clf_pipeline.predict(X_test_c)

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, y_pred),
        "Precision": precision_score(y_test_c, y_pred, zero_division=0),
        "Recall": recall_score(y_test_c, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test_c, y_pred, zero_division=0)
    })

classification_results_df = pd.DataFrame(classification_results)
classification_results_df = classification_results_df.sort_values(by="F1 Score", ascending=False)

# =========================
# 8. RUN REGRESSION
# =========================
regression_results = []

for name, model in regression_models.items():
    reg_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    reg_pipeline.fit(X_train_r, y_train_r)
    y_pred = reg_pipeline.predict(X_test_r)

    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))

    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_r, y_pred),
        "RMSE": rmse,
        "R2 Score": r2_score(y_test_r, y_pred)
    })

regression_results_df = pd.DataFrame(regression_results)
regression_results_df = regression_results_df.sort_values(by="RMSE", ascending=True)

# =========================
# 9. PRINT RESULTS
# =========================
print("\n===== Classification Model Comparison =====")
print(classification_results_df)

print("\n===== Regression Model Comparison =====")
print(regression_results_df)